# Big Data Lab 1 - Exercise 1
Binary Classifier: Will a user rate a movie >= 4?


In [ ]:
# Exercise 0: Prepare Movie Data
import findspark
import os
import sys
os.environ["JAVA_HOME"] = "C:\\Program Files\\Microsoft\\jdk-11.0.16.101-hotspot"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Initialize Spark session
spark = SparkSession.builder \
    .appName("BigData_Lab1") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

kafka_bootstrap_servers = "localhost:30090,localhost:30091,localhost:30092"

# Define schemas
movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True)
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True)
])

# Read movies from Kafka
df_movies_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_movies") \
    .option("startingOffsets", "earliest") \
    .load()
df_movies = df_movies_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), movies_schema).alias("data")) \
    .select("data.*")

# Read ratings from Kafka
df_ratings_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_ratings") \
    .option("startingOffsets", "earliest") \
    .load()
df_ratings = df_ratings_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), ratings_schema).alias("data")) \
    .select("data.*")

# Read tags from Kafka
df_tags_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "Lab1_tags") \
    .option("startingOffsets", "earliest") \
    .load()
df_tags = df_tags_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), tags_schema).alias("data")) \
    .select("data.*")

# Show schemas and sample data
df_movies.printSchema()
df_ratings.printSchema()
df_tags.printSchema()



## Data Preparation


In [ ]:
from pyspark.sql.functions import when, collect_list, concat_ws, avg, count, regexp_replace

# Target variable
df_ratings = df_ratings.withColumn("label", when(col("rating") >= 4, 1.0).otherwise(0.0))

# Aggregate tags
user_movie_tags = df_tags.groupBy("userId", "movieId").agg(concat_ws(" ", collect_list("tag")).alias("tags_text"))

# User aggregates
user_agg = df_ratings.groupBy("userId").agg(
    avg("rating").alias("user_avg_rating"),
    count("rating").alias("user_rating_count")
)

# Movie aggregates
movie_agg = df_ratings.groupBy("movieId").agg(
    avg("rating").alias("movie_avg_rating"),
    count("rating").alias("movie_rating_count")
)

# Join all features
df_data = df_ratings.join(df_movies, on="movieId", how="left") \
    .join(user_movie_tags, on=["userId", "movieId"], how="left") \
    .join(user_agg, on="userId", how="left") \
    .join(movie_agg, on="movieId", how="left")

# Fill missing values and clean genres/text
df_data = df_data.fillna({"tags_text": "", "title": "", "genres": ""})
df_data = df_data.withColumn("text_features", concat_ws(" ", col("title"), col("tags_text")))
df_data = df_data.withColumn("genres_space", regexp_replace("genres", "\\|", " "))

train_data, test_data = df_data.randomSplit([0.8, 0.2], seed=42)



## Build Pipeline


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, CountVectorizer, IDF, StopWordsRemover, Imputer, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Text Pipeline
tokenizer_text = Tokenizer(inputCol="text_features", outputCol="text_tokens")
remover_text = StopWordsRemover(inputCol="text_tokens", outputCol="text_clean")
cv_text = CountVectorizer(inputCol="text_clean", outputCol="text_tf", vocabSize=1000)
idf_text = IDF(inputCol="text_tf", outputCol="text_tfidf")

# Categorical Pipeline (Genres)
tokenizer_genres = Tokenizer(inputCol="genres_space", outputCol="genres_tokens")
cv_genres = CountVectorizer(inputCol="genres_tokens", outputCol="genres_vec")

# Numeric Pipeline
numeric_cols = ["user_avg_rating", "user_rating_count", "movie_avg_rating", "movie_rating_count"]
imputer = Imputer(inputCols=numeric_cols, outputCols=[c+"_imputed" for c in numeric_cols])

# Assembler & Classifier
assembler = VectorAssembler(
    inputCols=["text_tfidf", "genres_vec"] + [c+"_imputed" for c in numeric_cols],
    outputCol="features",
    handleInvalid="keep"
)
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

pipeline = Pipeline(stages=[
    tokenizer_text, remover_text, cv_text, idf_text,
    tokenizer_genres, cv_genres,
    imputer, assembler, lr
])

model = pipeline.fit(train_data)
predictions = model.transform(test_data)



## Evaluation


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC
evaluator_auc = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="label", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

# F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="label", metricName="f1")
f1 = evaluator_f1.evaluate(predictions)

print(f"AUC: {auc}")
print(f"F1 Score: {f1}")

# Confusion Matrix
print("Confusion Matrix:")
predictions.groupBy("label", "prediction").count().show()



## Top Features Extraction


In [ ]:
# Extract Coefficients and Vocabulary
lr_model = model.stages[-1]
coefficients = lr_model.coefficients.toArray()

# Extract vocabularies from CountVectorizers
text_vocab = model.stages[2].vocabulary  # cv_text
genres_vocab = model.stages[5].vocabulary # cv_genres
numeric_features = [c+"_imputed" for c in numeric_cols]

feature_names = text_vocab + genres_vocab + numeric_features
feat_coefs = list(zip(feature_names, coefficients))
feat_coefs.sort(key=lambda x: x[1])

print("Top 10 Negative Signals:")
for f, c in feat_coefs[:10]:
    print(f"{f}: {c:.4f}")

print("\nTop 10 Positive Signals:")
for f, c in feat_coefs[-10:][::-1]:
    print(f"{f}: {c:.4f}")

